# Data Visualizations: EuroConnect Telecom Customer Churn

This notebook generates key visualizations for the Customer Churn Prediction project.
It loads data and model outputs from Google Cloud Storage and produces charts covering:

1. **Churn Distribution** -- class balance in the training data
2. **Model Performance Comparison** -- Logistic Regression vs Random Forest
3. **Prediction Probability Distribution** -- histogram of churn probabilities
4. **Feature Importance / Churn by Key Features** -- tenure, contract type, internet service
5. **Business Impact** -- estimated monthly revenue at risk from predicted churners
6. **Confusion Matrix Heatmap** -- classification quality at a glance

In [ ]:
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage

warnings.filterwarnings('ignore', category=FutureWarning)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

# ---------- GCS configuration ----------
BUCKET_NAME = 'greencart-churn-regression'
PROJECT_ID  = 'mlip-485615'

gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(BUCKET_NAME)

def load_csv_from_gcs(blob_path: str) -> pd.DataFrame:
    """Download a CSV blob from GCS and return a DataFrame."""
    blob = bucket.blob(blob_path)
    csv_bytes = blob.download_as_bytes()
    return pd.read_csv(io.BytesIO(csv_bytes))

print('GCS client initialized.')

## Load Data from GCS

We load three files:
- **Original dataset** (`inputs/telco_customer_churn.csv`) -- for EDA-style charts
- **Predictions** (`outputs/predictions.csv`) -- model predictions with probabilities
- **Training metrics** (`outputs/training_metrics.csv`) -- per-model performance metrics

In [ ]:
# Load the original dataset
df = load_csv_from_gcs('inputs/telco_customer_churn.csv')
print(f'Original dataset: {df.shape[0]} rows, {df.shape[1]} columns')

# Clean TotalCharges (blank strings -> NaN -> 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Encode Churn to binary if stored as Yes/No
if not pd.api.types.is_numeric_dtype(df['Churn']):
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f'Churn rate: {df.Churn.mean()*100:.1f}%')
df.head()

In [ ]:
# Load predictions
predictions_df = load_csv_from_gcs('outputs/predictions.csv')
print(f'Predictions: {predictions_df.shape[0]} rows, {predictions_df.shape[1]} columns')
print(f'Columns: {list(predictions_df.columns)}')
predictions_df.head()

In [ ]:
# Load training metrics
metrics_df = load_csv_from_gcs('outputs/training_metrics.csv')
print(f'Training metrics: {metrics_df.shape[0]} rows, {metrics_df.shape[1]} columns')
metrics_df

---
## 1. Churn Distribution

A simple bar chart showing the class balance in the original training data.
The dataset is moderately imbalanced (~73% not churned, ~27% churned).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Bar chart: absolute counts ---
churn_counts = df['Churn'].value_counts().sort_index()
labels = ['Not Churned (0)', 'Churned (1)']
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(labels, churn_counts.values, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 80, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Churn Distribution (Counts)')
axes[0].set_ylim(0, churn_counts.max() * 1.15)

# --- Pie chart: proportions ---
axes[1].pie(
    churn_counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05),
    textprops={'fontsize': 12}
)
axes[1].set_title('Churn Distribution (Proportions)')

plt.suptitle('Customer Churn Distribution -- EuroConnect Telecom', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Total customers: {len(df):,}')
print(f'Churned: {churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.1f}%)')
print(f'Not churned: {churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.1f}%)')

---
## 2. Model Performance Comparison

A grouped bar chart comparing Logistic Regression and Random Forest
across all key classification metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC).

In [ ]:
# The training_metrics.csv is expected to have a 'Metric' column and model columns.
# Adjust parsing based on the actual structure.
# Common formats:
#   Format A: Metric, Logistic Regression, Random Forest  (wide)
#   Format B: Metric, Model, Value                        (long)
#   Format C: Metric, Training, Validation                (from baseline_metrics.csv style)
#
# We handle the most likely format and fall back gracefully.

print('Training metrics columns:', list(metrics_df.columns))
print()
print(metrics_df.to_string(index=False))

In [ ]:
# --- Parse metrics into a uniform format for plotting ---
# We expect either wide-format (model names as columns) or a Metric column as index.
# Detect format and normalise to: dict  metric_name -> {model_name: value}

def parse_metrics(mdf: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame with index = metric names, columns = model names.
    Handles several CSV layouts produced by the training pipeline.
    """
    # If first column looks like metric names, set it as index
    first_col = mdf.columns[0]
    if mdf[first_col].dtype == object and mdf[first_col].str.contains('Accuracy|F1|Recall|Precision|AUC', case=False).any():
        mdf = mdf.set_index(first_col)
    
    # Ensure all values are numeric
    mdf = mdf.apply(pd.to_numeric, errors='coerce')
    return mdf

metrics_parsed = parse_metrics(metrics_df.copy())
print('Parsed metrics:')
print(metrics_parsed.round(4).to_string())

# Identify model columns (exclude 'Training' if 'Validation' is present, to show val only)
model_columns = [c for c in metrics_parsed.columns if c not in ('Training',)]
if not model_columns:
    model_columns = list(metrics_parsed.columns)
print(f'\nModel columns for comparison chart: {model_columns}')

In [ ]:
# Grouped bar chart -- model performance comparison
metrics_to_plot = metrics_parsed[model_columns]
metric_names = list(metrics_to_plot.index)
n_metrics = len(metric_names)
n_models = len(model_columns)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(n_metrics)
width = 0.8 / n_models
palette = sns.color_palette('Set2', n_models)

for i, model in enumerate(model_columns):
    offset = (i - (n_models - 1) / 2) * width
    values = metrics_to_plot[model].values
    bars = ax.bar(x + offset, values, width, label=model, color=palette[i], edgecolor='black', linewidth=0.5)
    # Add value labels
    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison -- Logistic Regression vs Random Forest', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11, loc='upper right')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='Random baseline')

plt.tight_layout()
plt.show()

---
## 3. Prediction Probability Distribution

Histogram showing the distribution of predicted churn probabilities.
A well-calibrated model should separate the two classes with
distinct peaks near 0 and 1.

In [ ]:
# Identify the probability column in predictions_df
# Common names: 'churn_probability', 'predicted_probability', 'probability', 'Churn_Probability'
prob_col = None
for candidate in ['churn_probability', 'Churn_Probability', 'predicted_probability',
                   'probability', 'Probability', 'predicted_proba', 'pred_proba']:
    if candidate in predictions_df.columns:
        prob_col = candidate
        break

# Fallback: pick the first float column that looks like probabilities (values in 0-1)
if prob_col is None:
    for col in predictions_df.select_dtypes(include=[np.number]).columns:
        if predictions_df[col].between(0, 1).all():
            prob_col = col
            break

print(f'Probability column: {prob_col}')

# Also identify prediction label column
pred_col = None
for candidate in ['predicted_churn', 'Predicted_Churn', 'prediction', 'Prediction',
                   'predicted_label', 'Churn_Prediction', 'churn_prediction']:
    if candidate in predictions_df.columns:
        pred_col = candidate
        break

if pred_col is None:
    for col in predictions_df.columns:
        if predictions_df[col].isin([0, 1]).all() and col != prob_col:
            pred_col = col
            break

print(f'Prediction column: {pred_col}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

if prob_col is not None:
    probs = predictions_df[prob_col]
    
    # If we have actual labels, colour by them
    actual_col = None
    for candidate in ['Churn', 'actual_churn', 'Actual_Churn', 'actual', 'label']:
        if candidate in predictions_df.columns:
            actual_col = candidate
            break
    
    if actual_col is not None:
        for label, colour, name in [(0, '#2ecc71', 'Not Churned'), (1, '#e74c3c', 'Churned')]:
            subset = probs[predictions_df[actual_col] == label]
            ax.hist(subset, bins=30, alpha=0.6, color=colour, label=name, edgecolor='black', linewidth=0.5)
        ax.legend(fontsize=11)
    else:
        ax.hist(probs, bins=30, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Predicted Churn Probability', fontsize=12)
    ax.set_ylabel('Number of Customers', fontsize=12)
    ax.set_title('Distribution of Predicted Churn Probabilities', fontsize=14, fontweight='bold')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='Decision threshold (0.5)')
    ax.legend(fontsize=11)
    
    # Summary statistics
    predicted_churners = (probs >= 0.5).sum()
    print(f'Predicted churners (prob >= 0.5): {predicted_churners:,} / {len(probs):,} ({predicted_churners/len(probs)*100:.1f}%)')
    print(f'Mean probability: {probs.mean():.3f}')
    print(f'Median probability: {probs.median():.3f}')
else:
    ax.text(0.5, 0.5, 'No probability column found in predictions.csv',
            ha='center', va='center', transform=ax.transAxes, fontsize=14)

plt.tight_layout()
plt.show()

---
## 4. Feature Importance / Churn by Key Features

Three subplots showing churn rates broken down by the most predictive features
identified during model training: **tenure**, **contract type**, and **internet service**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- 4a. Churn rate by Tenure bins ---
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12 mo', '13-24 mo', '25-48 mo', '49-72 mo']
)
tenure_churn = df.groupby('tenure_group', observed=True)['Churn'].mean() * 100

bars = axes[0].bar(tenure_churn.index, tenure_churn.values, color='#3498db', edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, tenure_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[0].set_xlabel('Tenure Group')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Tenure')
axes[0].set_ylim(0, tenure_churn.max() * 1.2)

# --- 4b. Churn rate by Contract type ---
contract_churn = df.groupby('Contract')['Churn'].mean() * 100
contract_order = ['Month-to-month', 'One year', 'Two year']
contract_churn = contract_churn.reindex(contract_order)
contract_colors = ['#e74c3c', '#f39c12', '#2ecc71']

bars = axes[1].bar(contract_churn.index, contract_churn.values, color=contract_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, contract_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylim(0, contract_churn.max() * 1.2)

# --- 4c. Churn rate by Internet Service ---
internet_churn = df.groupby('InternetService')['Churn'].mean() * 100
internet_order = ['Fiber optic', 'DSL', 'No']
internet_churn = internet_churn.reindex(internet_order)
internet_colors = ['#e74c3c', '#f39c12', '#2ecc71']

bars = axes[2].bar(internet_churn.index, internet_churn.values, color=internet_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, internet_churn.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[2].set_xlabel('Internet Service')
axes[2].set_ylabel('Churn Rate (%)')
axes[2].set_title('Churn Rate by Internet Service')
axes[2].set_ylim(0, internet_churn.max() * 1.2)

plt.suptitle('Churn Rate by Key Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Clean up temp column
df.drop(columns=['tenure_group'], inplace=True)

---
## 5. Business Impact: Estimated Revenue at Risk

Using the predicted churners from the model, we estimate the **monthly revenue at risk**
by summing the `MonthlyCharges` of all customers predicted to churn.
This translates the model output into a concrete financial metric for stakeholders.

In [ ]:
# Merge predictions with MonthlyCharges from the original dataset
# We match on customerID if available, otherwise use index alignment.

if 'customerID' in predictions_df.columns and 'customerID' in df.columns:
    merged = predictions_df.merge(df[['customerID', 'MonthlyCharges']], on='customerID', how='left')
elif 'MonthlyCharges' in predictions_df.columns:
    merged = predictions_df.copy()
else:
    # Fallback: assume same row order
    merged = predictions_df.copy()
    if len(predictions_df) <= len(df):
        merged['MonthlyCharges'] = df['MonthlyCharges'].values[:len(predictions_df)]

# Identify churners from predictions
if pred_col is not None and 'MonthlyCharges' in merged.columns:
    churners = merged[merged[pred_col] == 1]
    non_churners = merged[merged[pred_col] == 0]
elif prob_col is not None and 'MonthlyCharges' in merged.columns:
    churners = merged[merged[prob_col] >= 0.5]
    non_churners = merged[merged[prob_col] < 0.5]
else:
    churners = pd.DataFrame()
    non_churners = pd.DataFrame()

print(f'Predicted churners:     {len(churners):,}')
print(f'Predicted non-churners: {len(non_churners):,}')

In [ ]:
if not churners.empty and 'MonthlyCharges' in churners.columns:
    revenue_at_risk = churners['MonthlyCharges'].sum()
    total_revenue = merged['MonthlyCharges'].sum()
    revenue_safe = non_churners['MonthlyCharges'].sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- 5a. Revenue pie chart ---
    axes[0].pie(
        [revenue_safe, revenue_at_risk],
        labels=['Retained Revenue', 'Revenue at Risk'],
        colors=['#2ecc71', '#e74c3c'],
        autopct='%1.1f%%',
        startangle=90,
        explode=(0, 0.05),
        textprops={'fontsize': 12}
    )
    axes[0].set_title('Monthly Revenue Breakdown', fontsize=13, fontweight='bold')

    # --- 5b. MonthlyCharges distribution: churners vs non-churners ---
    axes[1].hist(non_churners['MonthlyCharges'], bins=25, alpha=0.6,
                 color='#2ecc71', label='Not Churned', edgecolor='black', linewidth=0.5)
    axes[1].hist(churners['MonthlyCharges'], bins=25, alpha=0.6,
                 color='#e74c3c', label='Predicted Churned', edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Monthly Charges ($)', fontsize=12)
    axes[1].set_ylabel('Number of Customers', fontsize=12)
    axes[1].set_title('Monthly Charges: Churners vs Non-Churners', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=11)

    plt.suptitle('Business Impact -- Estimated Revenue at Risk from Predicted Churners',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary
    print(f'Total monthly revenue:          ${total_revenue:,.2f}')
    print(f'Revenue at risk (churners):      ${revenue_at_risk:,.2f} ({revenue_at_risk/total_revenue*100:.1f}%)')
    print(f'Revenue retained (non-churners): ${revenue_safe:,.2f} ({revenue_safe/total_revenue*100:.1f}%)')
    print(f'Avg monthly charge (churners):   ${churners["MonthlyCharges"].mean():,.2f}')
    print(f'Avg monthly charge (non-churn):  ${non_churners["MonthlyCharges"].mean():,.2f}')
    print(f'\nAnnualised revenue at risk: ${revenue_at_risk * 12:,.2f}')
else:
    print('Could not compute business impact -- MonthlyCharges not available in merged data.')

---
## 6. Confusion Matrix Heatmap

Visualise the confusion matrix from the model predictions.
If actual labels are available in the predictions file, we compute the matrix directly.
Otherwise we reconstruct it from the original dataset where possible.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Try to find actual and predicted labels
actual_col = None
for candidate in ['Churn', 'actual_churn', 'Actual_Churn', 'actual', 'label']:
    if candidate in predictions_df.columns:
        actual_col = candidate
        break

# If no actual column in predictions, try merging with original dataset on customerID
if actual_col is None and 'customerID' in predictions_df.columns and 'customerID' in df.columns:
    merged_labels = predictions_df.merge(df[['customerID', 'Churn']], on='customerID', how='inner')
    if len(merged_labels) > 0:
        actual_col = 'Churn'
        predictions_with_labels = merged_labels
    else:
        predictions_with_labels = predictions_df
else:
    predictions_with_labels = predictions_df

if actual_col is not None and pred_col is not None:
    y_true = predictions_with_labels[actual_col]
    y_pred = predictions_with_labels[pred_col]
    
    # Ensure binary encoding
    if not pd.api.types.is_numeric_dtype(y_true):
        y_true = (y_true == 'Yes').astype(int)
    if not pd.api.types.is_numeric_dtype(y_pred):
        y_pred = (y_pred == 'Yes').astype(int)
    
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Not Churned', 'Churned'],
        yticklabels=['Not Churned', 'Churned'],
        linewidths=1, linecolor='black',
        annot_kws={'size': 16, 'fontweight': 'bold'},
        ax=ax
    )
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('Actual Label', fontsize=12)
    ax.set_title('Confusion Matrix -- Model Predictions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print classification report
    print(classification_report(y_true, y_pred, target_names=['Not Churned', 'Churned']))
    
    # Print confusion matrix interpretation
    tn, fp, fn, tp = cm.ravel()
    print(f'True Negatives (correctly predicted not churned):  {tn:,}')
    print(f'False Positives (incorrectly predicted churned):   {fp:,}')
    print(f'False Negatives (missed churners):                 {fn:,}')
    print(f'True Positives (correctly predicted churned):      {tp:,}')
else:
    print('Cannot build confusion matrix: actual labels or predictions not found.')
    print(f'  actual_col = {actual_col}')
    print(f'  pred_col   = {pred_col}')
    print(f'  Available columns: {list(predictions_df.columns)}')

---
## Summary

This notebook produced six visualizations covering the full lifecycle of the churn prediction project:

| # | Visualization | Purpose |
|---|---|---|
| 1 | Churn Distribution | Understand class imbalance in the dataset |
| 2 | Model Performance Comparison | Compare LR vs RF across all metrics |
| 3 | Prediction Probability Distribution | Assess model confidence and calibration |
| 4 | Feature Importance / Churn by Key Features | Identify the most impactful drivers of churn |
| 5 | Business Impact -- Revenue at Risk | Translate predictions into financial impact |
| 6 | Confusion Matrix Heatmap | Evaluate classification accuracy in detail |

These visualizations can be shared with stakeholders to communicate both the technical
performance and business value of the churn prediction models.